# Relating two vehicles that never mention each other

`simple-demo.ipynb` builds the graph and asks it questions. This notebook is about the
one question that graph could not answer: *what in Apollo plays the role of the drone's
X?*

No Apollo `.sysml` file mentions a drone, so parsing produces no edge between the two
models and no community spanning them. Parsing is exact about what a file states, and
this is the cost of that: a resemblance is not stated anywhere, so there is nothing to
parse. The `analogy` build step adds the missing edges.

It does not invent an algorithm to do it. autograph already answers "which of these
resemble each other" in `corpus_graph.similarity_finding.SimilarityFinder` -- semantic
search and BM25 over one corpus, fused with reciprocal rank. That class runs unmodified
against the local container, the same way `enrich` already runs autograph's
`DataStorage`. Two things were arranged around it: the corpus it is handed is one
document per SysML **element** rather than per file, and its `module_doc_ids` restriction
is passed the *other* models, so the only edges it can build are the ones that cross.

In [1]:
from sysml import config, nl

db = config.db()

CROSS_MODEL = f'''
FOR r IN {config.RELATIONS}
  LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
  FILTER a.model != null AND b.model != null AND a.model != b.model
  COLLECT kind = r.type WITH COUNT INTO n
  RETURN {{kind, n}}'''

print("edges joining two different models:")
for row in db.aql.execute(CROSS_MODEL):
    print(f"  {row['n']:>4}  {row['kind']}")

edges joining two different models:


    37  SIMILAR_TO


`SIMILAR_TO` is the only one, and that is the point: every other edge in the graph was
read out of a file, and no file crosses a model boundary.

It is autograph's label, imported from `corpus_graph.naming`, deliberately not the
importer's. An analogy is not something a SysML file states, so it must not become a
`RELATED_TO` carrying a `relationship_type` -- that field means "an engineer wrote this
relation down", and a computed resemblance sitting in it would land in every count that
groups by it.

In [2]:
STRONGEST = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "{config.SIMILAR_TO}"
  LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
  SORT r.cosine DESC
  LIMIT 14
  RETURN {{a: a.name, am: a.model, b: b.name, bm: b.model,
           role: r.analogy_role, cosine: r.cosine}}'''

for r in db.aql.execute(STRONGEST):
    print(f"{r['cosine']:.3f}  {r['role']:<11} "
          f"{r['a']} ({r['am']})  ~  {r['b']} ({r['bm']})")

0.777  Port        ctrlPort (drone-logical)  ~  controlPort (apollo-11)
0.695  Part        LongDistanceDroneBattery (drone-logical)  ~  drone (drone-base)
0.673  Part        Drone (drone-logical)  ~  drone (drone-base)
0.667  Port        ctrlPort (drone-logical)  ~  ControlPort (apollo-11)
0.661  Part        DroneFlightControl (drone-logical)  ~  Drone (drone-base)
0.658  Port        ctrlPort (drone-logical)  ~  stageControlPort (apollo-11)
0.647  Part        engine1 (drone-logical)  ~  engines (apollo-11)
0.640  Part        engines (drone-logical)  ~  engines (apollo-11)
0.637  Part        engine2 (drone-logical)  ~  engine2 (apollo-11)
0.627  Port        powerCtrlPort (drone-logical)  ~  stageControlPort (apollo-11)
0.622  Part        engine4 (drone-logical)  ~  engine4 (apollo-11)
0.616  Part        drone (drone-base)  ~  droneBody4Engines (drone-logical)
0.613  Port        powerCtrlPort (drone-logical)  ~  controlPort (apollo-11)
0.610  Part        engine1 (drone-logical)  ~  engin

Read that as a similarity layer and not as an oracle. `ctrlPort ~ controlPort`,
`engines ~ engines` and `engine2 ~ engine2` are real correspondences. Further down the
same list are matches that are only topical, and `drone` from the drone-base model turns
up repeatedly because that model has seven comparable elements and its `drone` is
described in terms of everything it contains.

Two caps keep that in check: at most three counterparts per element, and at most two
elements pointing at any one counterpart. Without the second, one hub becomes the
analogue of a dozen unrelated things, which is the analogue of nothing.

Now ask it in English. AQLizer writes the traversal.

In [3]:
nl.instance().ask(
    "What does the drone's ctrlPort correspond to in the Apollo model?").show()

LLM provider initialized successfully.


Connecting to ArangoDB at http://localhost:8529 (timeout=300s)


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  What does the drone's ctrlPort correspond to in the Apollo model?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.name == "ctrlPort" AND e.model == "drone-logical"
     FOR other, edge IN 1..1 ANY e sysml_Relations
       FILTER edge.type == "SIMILAR_TO" AND other.model == "apollo-11"
       SORT edge.cosine DESC
       RETURN {element: e.name, counterpart: other.name, model: other.model, cosine: edge.cosine, at: CONCAT(other.source_file, ":", other.source_line)}

rows (3, first 3)
   {"element": "ctrlPort", "counterpart": "controlPort", "model": "apollo-11", "cosine": 0.7768328708023674, "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:195"}
   {"element": "ctrlPort", "counterpart": "ControlPort", "model": "apollo-11", "cosine": 0.6670945432679183, "at": "apollo-11-sysml-v2/Technical/TechnicalPortsPackage.sysml:33"}
   {"element": "ctrlPort", "counterpart": "stageControlPort", "model": "apollo-11", "cosine": 0.65826701939196

The retrieval path needed no changes at all. The local retriever expands over any edge
touching an entity it matched, so an analogy edge enters the context on its own, and its
`description` was written as a sentence for exactly that reason.

In [4]:
(await nl.retriever().ask_async(
    "What in the Apollo model plays a role like the drone ctrlPort?")).show()

RetrievalService initialized with chat_api_provider: openai, embedding_api_provider: openai


Processing local query: What in the Apollo model plays a role like the dro...


CACHE DISABLED (use_cache=False) - Skipping cache check


Creating LocalRetriever with provider: openai


Vector index for embedding already exists


...

LOCAL: model response length=513


Successfully processed LOCAL query


Cache check: use_cache=False, is_dict=True, from_cache=None, stream=False, query_type=LOCAL


CACHE DISABLED (use_cache=False) - Skipping cache write


Q  What in the Apollo model plays a role like the drone ctrlPort?

retrieved  20 documents, 61 edges, 40,480 chars of context

cited (6, first 6)
   {"cite": 1, "source": "models/DroneModelLogical.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/AstronautsPackage.sysml"}
   {"cite": 3, "source": "models/Drone_BaseArchitecture.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Technical/TechnicalPortsPackage.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 6, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}

A  ## Answer

In the Apollo model, the element that plays a role like the `ctrlPort` in the drone model is the `controlPort`. The relationship points to a strong similarity in role and function between `ctrlPort` in the drone-logical model and `controlPort` in the Apollo-11 model, with a description match at cosine 0.78[CITE:6]. Both elements are PortUsa

The more useful question for a systems engineer is the negative one. An analogy layer
that only reports matches is a search box; what it is actually good for is finding the
parts of one vehicle that nothing in the other resembles.

In [5]:
nl.instance().ask(
    "Which drone-logical PartUsage elements have no SIMILAR_TO edge at all? "
    "List the first eight with their source location.").show()

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  Which drone-logical PartUsage elements have no SIMILAR_TO edge at all? List the first eight with their source location.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.model == "drone-logical" AND e.entity_type == "PartUsage"
     LET counterparts = LENGTH(
       FOR r IN sysml_Relations
         FILTER r.type == "SIMILAR_TO" AND (r._from == e._id OR r._to == e._id)
         LIMIT 1 RETURN 1)
     FILTER counterparts == 0
     LIMIT 8
     RETURN {name: e.name, location: CONCAT(e.source_file, ":", e.source_line)}

rows (8, first 6)
   {"name": "battery", "location": "DroneModelLogical.sysml:77"}
   {"name": "charger", "location": "DroneModelLogical.sysml:80"}
   {"name": "body", "location": "DroneModelLogical.sysml:104"}
   {"name": "droneControlUnit", "location": "DroneModelLogical.sysml:108"}
   {"name": "safetyModule", "location": "DroneModelLogical.sysml:114"}
   {"name": "autonomousFlightModules", "location": "DroneModelLogical.sysml:115"}

## What this does and does not establish

It relates two models that never reference each other, it does so with autograph's own
similarity code rather than a bespoke one, and both read paths reach it -- one with the
AQL shown, the other with citations back to a source file.

What it is not: these are resemblances between *descriptions*, scored and thresholded.
Below the rows above, `engines ~ stages` is a real structural analogy and
`engines ~ AerospaceCompany` is noise, and nothing in the layer knows the difference.
The cosine is on every edge so a reader can judge, and the floor (0.55) and both caps
are constants at the top of `sysml/pipeline/analogy.py`.

One thing worth knowing before pointing this at a bigger corpus: autograph's corpus
layer truncates each document to `CHUNK_MAX_CHARS` -- 1200 tokens x 4 = 4,800 characters.
The largest file here is 58,902 bytes, so a document-level run would have compared 8% of
it. Raising `chunk_size` helps and does not solve it: the embedding model's context ends
at 8,192 tokens, and one vector for a 58 KB file of 296 requirements does not resemble
anything in particular no matter how much of it was read. Comparing elements sidesteps
both -- every element is far inside the bound.